In [1]:
import sys
sys.path.append('/data/bionets/og86asub/alternet-project/alternet/')
import pandas as pd
import alternet.data_preprocessing as preprocessing
from alternet.annotation import map_tf_ids
import alternet.annotation as annotation
import alternet.postprocessing as postprocessing

from alternet.inference import *
import yaml
import time
from typing import Any, Dict
import numpy as np

import seaborn as sns
import os.path as op
import os

def write_dict_to_yaml(data: Dict[str, Any], filepath: str):
    """Writes a Python dictionary to a YAML file."""

    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)


def add_gene_names_and_save(edge_data, condi, net_type, filepath):
    outfile = op.join(filepath, f"{condi}_{net_type}.tsv" )
    os.makedirs(filepath, exist_ok = True)

    # rename columns to fit a schema
    if 'target' in edge_data.columns and 'target_gene' not in edge_data.columns:
        edge_data = edge_data.rename(columns = {'target': 'target_gene'})


    if 'source_transcript' in edge_data.columns and 'source_gene_name' not in edge_data.columns:
        edge_data = postprocessing.add_transcript_names(edge_data, biomart, transcript_column = 'source_transcript')
    if 'source_gene' in edge_data.columns and 'source_gene_name' not in edge_data.columns:
        edge_data = postprocessing.add_gene_names(edge_data, biomart, gene_column = 'source_gene')
    if 'target_gene' in edge_data.columns and 'target_gene_name' not in edge_data.columns:
        edge_data = postprocessing.add_gene_names(edge_data, biomart, gene_column = 'target_gene')
        
    edge_data.to_csv(outfile)



/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numba/core/decorators.py:248: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [ ]:
results_path = "/data/bionets/og86asub/alternet-project/alternet-manuscript/results/"
data_path = "/data/bionets/og86asub/alternet-project/alternet/data/"
appris_path = "/data/bionets/og86asub/alternet-project/alternet/data/appris_data.appris.txt"
digger_path = "/data/bionets/og86asub/alternet-project/alternet/data/digger_data.csv"
biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
tf_list_path = '/data/bionets/og86asub/alternet-project/alternet/data/allTFs_hg38.txt'


os.makedirs(results_path, exist_ok=True)

appris_df = pd.read_csv(appris_path, sep='\t')
biomart = pd.read_csv(biomart_path, sep='\t')
tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)
tf_database = annotation.create_transcipt_annotation_database(tf_list=tf_list, appris_path= appris_path , digger_path=digger_path)
transcript_mapper = annotation.create_transcript_mapping(biomart)


runs = 10
conditions = ['DCM', 'NF', 'HCM']

for condi in conditions:

    magnet_path = op.join(data_path, f"{condi}_magnet_prefiltered_tpm.tsv")
    transcript_data_cp = pd.read_csv(magnet_path, sep='\t')
    gene_data_cp = transcript_data_cp.groupby('gene_id').sum()
    gene_data_cp = gene_data_cp.drop(columns={'transcript_id'})

    gene_data_cp.index.name = 'gene_id'
    gene_data_cp = gene_data_cp.reset_index()

    gene_target_names = list(tf_list['Gene stable ID'].unique())


    gene_data_cp = gene_data_cp.set_index('gene_id')
    gene_data_cp = gene_data_cp.T

    # scale data!!
    gene_data_cp_scaled = standardize_dataframe(gene_data_cp)

    transcript_data_cp = transcript_data_cp.set_index('transcript_id')
    transcript_data_cp = transcript_data_cp.drop('gene_id', axis=1)
    transcript_data_cp = transcript_data_cp.T

    # scale data!!
    transcript_data_cp_scaled = standardize_dataframe(transcript_data_cp)

    # compute isoform categories
    isoform_categories = postprocessing.isoform_categorization(transcript_data_cp, gene_data_cp, tf_list)
    gene_categories = postprocessing.get_gene_cases(isoform_categories)
    gene_to_transcript_mapping = annotation.create_filtered_gene_to_transcripts_mapping(biomart, gene_list = gene_data_cp_scaled.columns, transcript_list = transcript_data_cp_scaled.columns)


    # start = time.monotonic()
    # canonical_grn = inference(gene_data=gene_data_cp, tf_list=gene_target_names, target_names = 'all', n_runs=runs)
    # end = time.monotonic()
    # runtime = {'canonical': end-start}
    
    # canonical_grn.to_csv(op.join(results_path, f"{condi}_canonical.tsv"), header = True)

    # hybrid_data = create_hybrid_data(transcript_data_cp, gene_data_cp, tf_list)
    # hybrid_tf_names = list(tf_list['Transcript stable ID'].unique())
    # target_names = list(gene_data_cp.columns)

    # start = time.monotonic()
    # as_aware_grn = inference(gene_data=hybrid_data, tf_list=hybrid_tf_names, target_names=target_names, n_runs = runs)
    # runtime['as_aware'] = time.monotonic()-start

    # as_aware_grn.to_csv(op.join(results_path, f"{condi}_as_aware.tsv"), header = True) 
    # write_dict_to_yaml(runtime, op.join(results_path, f"{condi}_runtime.yaml"))

    start_postprocessing = time.monotonic()


    #filter aggregate

    as_aware_grn = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_as_aware.tsv", index_col=0)
    canonical_grn = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_canonical.tsv", index_col=0)


    as_aware_grn = postprocessing.map_transcript_to_gene(as_aware_grn, transcript_mapper)
    as_aware_grn = postprocessing.create_edge_key(as_aware_grn)
    canonical_grn = postprocessing.create_edge_key(canonical_grn, source_column='source', target_column='target')
    canonical_grn = canonical_grn.rename(columns={'source': 'source_gene'})


    # Categorize edges into gene-unique, isoform-unique, common
    common_edges = postprocessing.get_common_edges(canonical_grn, as_aware_grn)
    gene_unique, isoform_unique = postprocessing.get_diff(canonical_grn, as_aware_grn)

    filtering_tracker = {'as_aware_base': as_aware_grn.shape[0], 'canonical_base': canonical_grn.shape[0],   'gene_unique_base': gene_unique.shape[0], 'isoform_unique_base': isoform_unique.shape[0]}


    # remove low quality edges
    gene_unique, gene_unique_filter_1 = postprocessing.frequency_filter(gene_unique,  threshold_frequency=runs)
    isoform_unique, isoform_unique_filter_1 = postprocessing.frequency_filter(isoform_unique, threshold_frequency=runs)

    # remove implausible edges, i.e. isoform edges where there is a dominant/single isoform which
    # should have been in gene GRN as well.
    isoform_unique, isoform_unique_plausibility_filter  = postprocessing.plausibility_filtering(isoform_unique, isoform_categories)
    # Merge isoform exon/domain information and save
    isoform_unique = annotation.annotate_isoform_exclusive_edges(isoform_unique, tf_database, transcript_column='source_transcript')
    


    # remove implausible edges, i.e. edges where there is a single/dominant isoform which should have been in
    # isoform GRN as well.
    gene_unique, gene_unique_plausibility_filter = postprocessing.plausibility_filtering_gene_unique(gene_unique, gene_categories)
    gene_unique = annotation.annotate_gene_exclusive_edges(gene_unique, annotation_database=tf_database, gene_transcript_mapping=gene_to_transcript_mapping)


    # remainder is in both GRNs. Merge the gene information to the isoform information
    merged_edges = postprocessing.create_common_edge_dataframe(common_edges)
    filtering_tracker['common_base'] = merged_edges.shape[0]

    # Split into dominant/single isoform and ambigous (multiple isoforms)
    consistent, ambigous = postprocessing.split_by_isoform_category(merged_edges, gene_categories)
    # remove edges which have vastly different frequencies and importances from dataframe (edges in common dataframe where there is only one single or
    # a dominant isoform should have a consistent importance and frequency).
    consistent, consistent_freq_filer = postprocessing.frequency_filtering_common_edges_dominant(consistent, threshold_frequency = runs)
    consistent, common_dominant_filter = postprocessing.plausibility_filtering_common_edges_dominant(consistent)
    consistent = annotation.annotate_consistent_edges(consistent, tf_database, transcript_column='source_transcript')


    # From remaining ambigous dataframe, select those edges which are sig. more importantn in isoform data frame
    likely_isoform_specific, ambigous, common_isoform_likely_filter = postprocessing.find_likely_isoform_specific(ambigous)
    # add transcript annotation and save
    likely_isoform_specific = annotation.annotate_isoform_exclusive_edges(likely_isoform_specific, tf_database, transcript_column='source_transcript')

    
    # from remaining ambigous dataframe, select those where the gene edge is more important than any isoform edge.
    likely_gene_specific, ambigous, common_gene_likely_filter = postprocessing.find_likely_gene_specific(ambigous)
    likely_gene_specific = annotation.annotate_gene_exclusive_edges(likely_gene_specific, annotation_database=tf_database, gene_transcript_mapping=gene_to_transcript_mapping)
    
    # remaining edges are ambigous.
    ambigous, ambigous_freq_filter = postprocessing.frequency_filtering_common_edges_dominant(ambigous, threshold_frequency = runs)

    correlation_collector = annotation.compute_isoform_gene_correlations(transcript_data_cp_scaled, gene_data_cp_scaled, gene_to_transcript_mapping)

    # Compute absolute importance threshold based on all surviving edgeds
    all_importances = np.concatenate([isoform_unique.mean_importance, gene_unique.median_importance, likely_gene_specific.median_importance_ge, likely_isoform_specific.median_importance_te, consistent.median_importance_te, ambigous.median_importance_te])
    absolute_threshold = np.percentile(all_importances, q=80)

    isoform_unique, filter_info_iu = postprocessing.filter_importance(isoform_unique, absolute_treshold=absolute_threshold,importance_column='median_importance' )
    gene_unique, filter_info_gu = postprocessing.filter_importance(gene_unique, absolute_treshold=absolute_threshold,importance_column='median_importance' )
    likely_gene_specific, filter_info_lgu =  postprocessing.filter_importance(likely_gene_specific, absolute_treshold=absolute_threshold,importance_column='median_importance_ge' )
    likely_isoform_specific, filter_info_liu =  postprocessing.filter_importance(likely_isoform_specific, absolute_treshold=absolute_threshold,importance_column='median_importance_ge' )
    consistent, filter_info_c  =  postprocessing.filter_importance(consistent, absolute_treshold=absolute_threshold,importance_column='median_importance_te' )
    ambigous, filter_info_a  =  postprocessing.filter_importance(ambigous, absolute_treshold=absolute_threshold,importance_column='median_importance_te' )


    add_gene_names_and_save(isoform_unique, condi, 'unique_isoforms', results_path)
    add_gene_names_and_save(gene_unique, condi, 'unique_genes', results_path)
    add_gene_names_and_save(consistent, condi, 'consistent_both', results_path)
    add_gene_names_and_save(likely_isoform_specific, condi, 'likely_isoform_specific', results_path)
    add_gene_names_and_save(likely_gene_specific, condi, 'likely_gene_specific', results_path)
    add_gene_names_and_save(ambigous, condi, 'ambigous', results_path)



    filtering_tracker['gene_unique_filter_frequency'] = gene_unique_filter_1
    filtering_tracker['isoform_unique_filter_frequency'] = isoform_unique_filter_1
    filtering_tracker['isoform_unique_plausibility_filter'] = isoform_unique_plausibility_filter
    filtering_tracker['gene_unique_plausibility_filter'] = gene_unique_plausibility_filter

    filtering_tracker['common_consistent_frequency_filter'] = consistent_freq_filer
    filtering_tracker['common_consitent_dominant_filter'] = common_dominant_filter
    
    filtering_tracker['common_isoform_likely_filter'] = common_isoform_likely_filter
    filtering_tracker['common_gene_likely_filter'] = common_gene_likely_filter
    filtering_tracker['common_ambigous_frequency_filter'] = ambigous_freq_filter

    filtering_tracker['isoform_unique_importance_threshold'] = filter_info_iu
    filtering_tracker['gene_unique_importance_threshold'] = filter_info_gu
    filtering_tracker['likely_gene_specific_importance_threshold'] = filter_info_lgu
    filtering_tracker['likely_isofrom_specific_importance_threshold'] = filter_info_liu
    filtering_tracker['common_consistent_importance_threshold'] = filter_info_c
    filtering_tracker['common_ambigous_importance_threshold'] = filter_info_a
    
    write_dict_to_yaml(filtering_tracker, op.join(results_path, f"{condi}_filtering_tracker.yaml"))
    elapsed = {'postprocessing_elapsed': time.monotonic() - start_postprocessing}
    write_dict_to_yaml(elapsed, op.join(results_path, f"{condi}_runtime_postprocessing.yaml"))






/data/bionets/og86asub/alternet-project/alternet/src/alternet/annotation.py:91: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  digger = pd.read_csv(digger_path)
/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered i

In [ ]:
ambigous, ambigous_freq_filter = postprocessing.frequency_filtering_common_edges_dominant(ambigous, threshold_frequency = runs)


In [8]:
import pandas as pd
import numpy as np

In [87]:
biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
biomart = pd.read_csv(biomart_path, sep='\t')

appris_path = '/data/bionets/mi34qaba/SpliceAwareGRN/data/appris_data.appris.txt'
appris_df = pd.read_csv(appris_path, sep='\t')



In [67]:
ktmp = pd.read_csv("/data/bionets/mi34qaba/data/MAGNet/MAGNet_transcript/MAGNet_kallisto_tpms", sep = '\t')


In [ ]:
ktmp = ktmp.rename(columns={'target_id': 'transcript_id'})
names = [x.replace('_est_tpms', '') for x in ktmp.columns]
ktmp.columns = names


NameError: name 'biomart' is not defined

In [88]:

# Add gene ID
ktmp = ktmp.merge(biomart.loc[: ,['Transcript stable ID', 'Gene stable ID']], left_on ='transcript_id', right_on ='Transcript stable ID')
ktmp = ktmp.drop(columns=['Transcript stable ID'])
ktmp = ktmp.rename(columns={"Gene stable ID":'gene_id'})

# subset to protein coding transcripts
ktmp = ktmp[ktmp.transcript_id.isin(appris_df[appris_df['Transcript type']=='protein_coding']['Transcript ID'])]


In [125]:
nonzero_count = np.count_nonzero(ktmp.values, axis=1)
frac = nonzero_count/ktmp.shape[1]
ktmp2 = ktmp.iloc[np.where(frac>0.50)[0]]



In [126]:
ktmp2.shape

(41395, 368)

In [127]:
ktmp2[~ktmp2.transcript_id.isin(maj.transcript_id)]

,transcript_id,SRR10676821,SRR10676822,SRR10676823,SRR10676824,SRR10676825,SRR10676826,SRR10676827,SRR10676828,SRR10676829,...,SRR10677178,SRR10677179,SRR10677180,SRR10677181,SRR10677182,SRR10677183,SRR10677184,SRR10677185,SRR10677186,gene_id
257,ENST00000643797,0.000000,0.000000,0.359728,0.000000,0.649254,0.000000,0.210922,0.000000,0.000000,...,0.174156,1.907470,4.082820,0.210496,0.369708,0.000000,0.000000,0.137721,1.769320,ENSG00000233276
1126,ENST00000361641,0.586338,0.659426,0.239397,0.266312,0.767723,0.000000,0.698817,0.780363,0.957973,...,0.333257,0.000000,0.648536,0.000000,0.534631,0.706167,0.496130,0.604312,0.119019,ENSG00000198860
1132,ENST00000647465,0.223349,0.304585,0.183878,0.000000,0.269676,1.287170,0.432725,0.709702,0.599334,...,0.000000,0.636124,0.140022,0.452664,0.925029,0.651252,0.330928,0.000000,0.668654,ENSG00000198860
1144,ENST00000358170,0.000006,0.011736,0.208666,0.457645,0.204763,0.000000,0.019586,1.404830,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.539244,0.000000,0.000000,0.000000,ENSG00000117335
1145,ENST00000354848,10.782600,0.906735,10.694700,2.669520,1.755410,0.000000,7.641400,3.011320,0.000001,...,4.270940,0.000000,0.427449,0.000000,0.000000,5.730670,6.296110,2.585520,0.000000,ENSG00000117335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166912,ENST00000359512,1.694600,0.186972,1.247890,0.568013,0.835642,0.000000,1.164890,1.555980,0.593583,...,0.999170,0.000000,0.386572,0.000000,0.179736,0.621722,0.926521,0.679563,0.076583,ENSG00000182484
169102,ENST00000436466,0.025797,0.024539,0.000000,0.000000,0.018499,0.000000,0.015889,0.040423,0.082446,...,0.025100,0.000000,0.012874,0.000000,0.000000,0.079912,0.000000,0.026899,0.000000,ENSG00000223731
171439,ENST00000522557,0.142755,0.074933,0.000000,0.121709,0.101315,0.099346,0.000000,0.337420,0.230882,...,0.513614,0.150015,0.338667,0.128717,0.000000,0.055904,0.264140,1.169410,0.000000,ENSG00000124097
172249,ENST00000395675,0.235977,0.079020,0.225104,0.116381,0.173702,0.000000,0.099064,0.073720,0.019465,...,0.036184,0.000000,0.055751,0.000000,0.000000,0.000000,0.062216,0.039616,0.000000,ENSG00000240445


In [118]:
ktmp[ktmp.gene_id == 'ENSG00000180071']

,transcript_id,SRR10676821,SRR10676822,SRR10676823,SRR10676824,SRR10676825,SRR10676826,SRR10676827,SRR10676828,SRR10676829,...,SRR10677178,SRR10677179,SRR10677180,SRR10677181,SRR10677182,SRR10677183,SRR10677184,SRR10677185,SRR10677186,gene_id
81770,ENST00000602295,0.092492,0.040614,0.084843,0.142823,0.062686,0.284741,0.098928,0.227802,0.125720,...,0.057707,0.224606,0.151398,0.203065,0.439322,0.035861,0.191838,0.206558,0.191033,ENSG00000180071
81771,ENST00000399703,0.357357,0.052489,0.419418,0.192821,0.118320,0.000000,0.415674,0.338804,0.091761,...,0.321425,0.085038,0.032196,0.017116,0.192424,0.077643,0.227482,0.301441,0.004909,ENSG00000180071


In [61]:
maj = pd.read_csv("/data/bionets/mi34qaba/data/MAGNet/MAGNet_transcript/MAGNet_tpms.csv", sep = ',')

In [110]:
maj[~maj.transcript_id.isin(ktmp2.transcript_id)]

,Unnamed: 0,transcript_id,gene_id,C00039,C00055,C00074,C00085,C00105,C00120,C00132,...,P01626,P01627,P01628,P01629,P01630,P01631,P01634,P01635,P01639,P01640
0,0,ENST00000473530,ENSG00000244682,2.276150,2.040930,1.564230,2.629880,3.224940,0.081621,3.577190,...,2.249030,0.754787,0.000000,1.594620,0.000000,0.031552,1.032080,2.658050,1.475990,0.012955
1,1,ENST00000467903,ENSG00000244682,2.167410,2.788660,3.829120,3.177140,4.211690,0.176553,6.959880,...,4.777670,2.296700,0.256739,7.109500,0.226315,0.453683,1.436180,2.298990,3.807010,0.155549
2,2,ENST00000508651,ENSG00000244682,1.950990,2.569730,3.989870,3.134920,5.018950,0.110165,4.027630,...,4.473280,1.117980,0.185469,3.784620,0.146539,0.085379,1.271050,1.629250,3.130670,0.000000
3,3,ENST00000626647,ENSG00000280938,0.049179,0.019637,0.017334,0.489663,0.000000,0.105393,0.517216,...,0.494979,3.469800,6.911700,1.803350,1.116030,0.850385,0.507154,0.000000,1.293200,1.458430
4,4,ENST00000633377,ENSG00000282181,3.231750,1.134710,3.935840,0.548367,0.007815,0.319113,3.020600,...,2.497850,0.987592,0.007706,0.507429,0.257754,0.544622,0.007459,1.090220,0.713997,1.031000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39306,39306,ENST00000640496,ENSG00000288373,0.171614,0.029266,0.338498,0.064711,0.063730,0.061593,0.247538,...,0.243201,0.055047,0.028341,0.029264,0.041927,0.032738,0.019335,0.034304,0.172885,0.019898
39307,39307,ENST00000642506,ENSG00000285059,0.479574,0.250078,1.438900,0.648053,0.522874,0.197844,0.570072,...,0.726277,0.521375,0.097360,0.753155,0.768252,0.059618,0.385886,0.475839,0.421306,0.199642
39309,39309,ENST00000562942,ENSG00000180071,0.600792,0.096192,0.531721,0.429547,0.395359,1.139980,0.555878,...,0.220314,3.903400,2.071020,0.256167,2.381600,4.226480,0.145904,0.275122,0.947478,0.153121
39310,39310,ENST00000569097,ENSG00000197081,0.821758,0.377072,0.413925,0.907076,0.476571,0.098923,0.395161,...,0.957061,0.418827,0.047140,0.320178,0.066159,0.132223,0.285653,0.631274,0.421508,0.062954


In [57]:
ma = pd.read_csv('/data/bionets/og86asub/alternet-project/alternet/data/DCM_magnet_prefiltered_tpm.tsv', sep='\t')

In [52]:
magnet = pd.read_csv('/data/bionets/og86asub/alternet-project/alternet/data/magnet_prefiltered_tpm.tsv', sep='\t')

In [62]:
ktmp[ktmp.target_id.isin(maj.transcript_id)]

,target_id,SRR10676821_est_tpms,SRR10676822_est_tpms,SRR10676823_est_tpms,SRR10676824_est_tpms,SRR10676825_est_tpms,SRR10676826_est_tpms,SRR10676827_est_tpms,SRR10676828_est_tpms,SRR10676829_est_tpms,...,SRR10677177_est_tpms,SRR10677178_est_tpms,SRR10677179_est_tpms,SRR10677180_est_tpms,SRR10677181_est_tpms,SRR10677182_est_tpms,SRR10677183_est_tpms,SRR10677184_est_tpms,SRR10677185_est_tpms,SRR10677186_est_tpms
238,ENST00000473530,2.276150,2.040930,1.564230,2.629880,3.224940,0.081621,3.577190,3.275500,0.692927,...,2.249030,0.754787,0.000000,1.594620,0.000000,0.031552,1.032080,2.658050,1.475990,0.012955
241,ENST00000467903,2.167410,2.788660,3.829120,3.177140,4.211690,0.176553,6.959880,3.485350,2.761340,...,4.777670,2.296700,0.256739,7.109500,0.226315,0.453683,1.436180,2.298990,3.807010,0.155549
243,ENST00000508651,1.950990,2.569730,3.989870,3.134920,5.018950,0.110165,4.027630,1.973540,1.143040,...,4.473280,1.117980,0.185469,3.784620,0.146539,0.085379,1.271050,1.629250,3.130670,0.000000
252,ENST00000626647,0.049179,0.019637,0.017334,0.489663,0.000000,0.105393,0.517216,0.029199,0.026186,...,0.494979,3.469800,6.911700,1.803350,1.116030,0.850385,0.507154,0.000000,1.293200,1.458430
254,ENST00000633377,3.231750,1.134710,3.935840,0.548367,0.007815,0.319113,3.020600,1.936820,2.938650,...,2.497850,0.987592,0.007706,0.507429,0.257754,0.544622,0.007459,1.090220,0.713997,1.031000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185014,ENST00000569097,0.821758,0.377072,0.413925,0.907076,0.476571,0.098923,0.395161,0.379031,0.214313,...,0.957061,0.418827,0.047140,0.320178,0.066159,0.132223,0.285653,0.631274,0.421508,0.062954
185315,ENST00000602699,0.050290,0.014330,0.035171,0.041823,0.037060,0.000000,0.065104,0.016308,0.026477,...,0.044029,0.000000,0.000000,0.025864,0.024941,0.048181,0.038492,0.051400,0.017281,0.014494
186340,ENST00000594408,2.056060,1.068780,0.909087,1.211280,1.235250,0.022664,1.194140,1.571050,0.650706,...,1.833670,0.982506,0.017389,0.592679,0.000000,0.043223,0.683917,1.783930,1.038960,0.000000
186752,ENST00000510091,0.397195,0.139399,0.137387,0.238294,0.456423,0.055239,0.127514,0.330749,0.091594,...,1.001040,0.285062,0.049527,0.056683,0.000000,0.000000,0.040746,0.233459,0.148919,0.000000


In [11]:
nonzero_count = np.count_nonzero(ma.values, axis=1)
frac = nonzero_count/ma.shape[1]


In [43]:
dcmun = pd.read_csv('/data/bionets/og86asub/alternet-project/alternet-manuscript/results2/DCM_likely_isoform_specific.tsv', sep=',')

In [45]:
dcmun[0:40]['median_importance_te'].mean()

46.722585897532255

In [46]:
highly = dcmun[~dcmun.source_transcript.isin(list(ma.iloc[np.where(frac<0.30)[0]]['transcript_id']))]['target_gene']

In [47]:
for h in highly.unique()[0:200]:
    print(h)

ENSG00000080371
ENSG00000129116
ENSG00000161016
ENSG00000272333
ENSG00000128708
ENSG00000137714
ENSG00000198793
ENSG00000214655
ENSG00000062582
ENSG00000198752
ENSG00000099991
ENSG00000141522
ENSG00000160305
ENSG00000099381
ENSG00000139637
ENSG00000134375
ENSG00000070061
ENSG00000112514
ENSG00000183023
ENSG00000179526
ENSG00000138592
ENSG00000240972
ENSG00000074071
ENSG00000118640
ENSG00000178188
ENSG00000134571
ENSG00000176058
ENSG00000038274
ENSG00000138468
ENSG00000005206
ENSG00000014824
ENSG00000186566
ENSG00000005007
ENSG00000121406
ENSG00000163565
ENSG00000063241
ENSG00000172915
ENSG00000185359
ENSG00000159069
ENSG00000109736
ENSG00000111716
ENSG00000104936
ENSG00000138138
ENSG00000179832
ENSG00000125995
ENSG00000137497
ENSG00000186395
ENSG00000010256
ENSG00000062485
ENSG00000068028
ENSG00000104866
ENSG00000137962
ENSG00000132694
ENSG00000082898
ENSG00000128731
ENSG00000136153
ENSG00000176978
ENSG00000138100
ENSG00000133317
ENSG00000129968
ENSG00000104915
ENSG00000090339
ENSG0000